In [1]:
%pwd

'd:\\DoAn\\Video_Anomaly_Detection\\AI\\notebooks'

In [2]:
%cd   d:\\DoAn\\Video_Anomaly_Detection\

d:\DoAn\Video_Anomaly_Detection


In [3]:
import re
import os
from AI.src.utils.misc import draw_anomaly_graph , calculate_iou , Overlap_ratio
from AI.src.data.dataset import VADFrameLevelDataset
from Web.src.be.src.utils.video_utils import find_anomaly_regions


Initializing DLL path for Windows


In [4]:
def parse_pred_file(file_path):
    """
    Parse the prediction result file that contains scores and ground truth labels
    
    Returns:
        all_scores: List of prediction score lists
        all_anomaly_ranges: List of lists of anomaly regions (start, end) for each video
        video_indices: List of video indices
    """
    all_scores = []
    all_labels = []
    all_anomaly_ranges = []
    video_indices = []
    
    with open(file_path, 'r') as f:
        content = f.read()
    
    # Find all instances of the pattern [scores],[labels],index
    pattern = r'\[(.*?)\],\[(.*?)\],(\d+)'
    matches = re.findall(pattern, content)
    
    for match in matches:
        scores_str, labels_str, video_idx = match
        
        # Parse scores
        scores = [float(s.strip()) for s in scores_str.split(',')]
        
        # Parse labels
        labels = [int(l.strip()) for l in labels_str.split(',')]
        
        # Find anomaly regions from labels
        anomaly_ranges = []
        start = None
        
        for i, label in enumerate(labels):
            if label == 1 and start is None:
                start = i
            elif label == 0 and start is not None:
                anomaly_ranges.append((start, i - 1))
                start = None
        
        # If there's an anomaly that extends to the end of the video
        if start is not None:
            anomaly_ranges.append((start, len(labels) - 1))
        
        all_scores.append(scores)
        all_labels.append(labels)
        all_anomaly_ranges.append(anomaly_ranges)
        video_indices.append(int(video_idx))
    
    return all_scores, all_anomaly_ranges, video_indices

In [5]:
dataset_root = r"D:\DoAn\Video_Anomaly_Detection"
annotation_file = "label.csv"

dataset = VADFrameLevelDataset(
    root = dataset_root,
    annotation=annotation_file,
    loader="v4"
)
video_paths = dataset._VADFrameLevelDataset__annotation["path"].tolist()
video_names = {i: path.replace(".pt", "") for i, path in enumerate(video_paths)}

In [ ]:
video_scores,labels,video_indices = parse_pred_file(rf"D:\DoAn\Video_Anomaly_Detection\pred_result.txt")
# output_dir = rf"D:\DoAn\Video_Anomaly_Detection\plot\fig1"

# os.makedirs(output_dir, exist_ok=True)

output_dir_fig2 = rf"D:\DoAn\Video_Anomaly_Detection\plot\fig3"
os.makedirs(output_dir_fig2, exist_ok=True)

##fig1

In [7]:
# for i, video_idx in enumerate(video_indices):
#     pred = video_scores[i]
#     video_anomaly_ranges = labels[i]
#     video_name = video_names.get(video_idx, f"Video_{video_idx}")
    
#     # Replace slashes with underscores to create a valid filename
#     safe_video_name = video_name.replace('/', '_').replace('\\', '_')
    
#     print(f"\nProcessing video: {video_name} (index: {video_idx})")
#     save_path = os.path.join(output_dir, f"fig1_{safe_video_name}.png")

#     draw_anomaly_graph(preds=pred,
#                       anomaly_ranges=video_anomaly_ranges,
#                       video_name=video_name,  # Keep original name for display
#                       save_path=save_path
#                       )

##fig2

In [ ]:
# Thông số cho phát hiện peak
os.environ["HEIGHT"] = str(0.9)
os.environ["THRESHOLD"] = str(None) 
os.environ["DISTANCE"] = str(None)
os.environ["PROMINENCE"] = str(0.00001)
os.environ["WIDTH"] = str(3)
os.environ["WLEN"] = str(None)
os.environ["REL_HEIGHT"] = str(0.8)
os.environ["PLATEAU_SIZE"] = str(None)

os.environ["MERGE_GAP"] = str(5)


# Tạo thư mục để lưu biểu đồ


sum = 0
sum_1 = 0
for i, video_idx in enumerate(video_indices):
    pred = video_scores[i]
    video_anomaly_ranges = labels[i]
    video_name = video_names.get(video_idx, f"Video_{video_idx}")

    safe_video_name = video_name.replace('/', '_').replace('\\', '_')
    
    # print(f"\nProcessing video: {video_name} (index: {video_idx})")
    
    # 1. Smooth the data
    detected_regions, processed_scores, peaks = find_anomaly_regions(pred,
                                                                     )
    # 2. Calculate IOU and Overlap ratio
    iou = calculate_iou(video_anomaly_ranges, detected_regions)
    overlap_ratio = Overlap_ratio(video_anomaly_ranges, detected_regions)

    print(f"Ground truth anomaly ranges: {video_anomaly_ranges}")
    print(f"Detected anomaly regions: {detected_regions}")
    # Print peak information
    # print(f"Video {video_name} has {len(peaks)} detected peaks")
    # if len(peaks) > 0:
    #     print(f"Peaks at frames: {peaks}")
    #     print(f"Detected anomaly regions: {detected_regions}")
    
    # 3. Draw graph with ground truth and detected anomaly regions
    save_path = os.path.join(output_dir_fig2, f"fig2_{safe_video_name}.png")
    
    
    sum += iou 
    sum_1 += overlap_ratio 
    print(f"sum iou: {sum}")
    print(f"sum overlap ratio: {sum_1}")
    draw_anomaly_graph(
        # preds=pred,
        anomaly_ranges=video_anomaly_ranges,
        video_name=video_name,  # Keep original name for display
        save_path=save_path,
        smooth_pred=processed_scores,
        smooth_label="Smoothed pred",
        additional_anomaly_ranges=detected_regions,
        additional_anomaly_color="green",
        peaks=peaks,
        iou= True,
        Overlap = True,
    )
    


PeakDetector(height=0.9, threshold=None, distance=None, prominence=1e-05, width=3, wlen=None, rel_height=0.8, plateau_size=None)
Ground truth anomaly ranges: [(270, 600)]
Detected anomaly regions: [(14, 163), (178, 360), (373, 838)]
sum iou: 0.3933415536374846
sum overlap ratio: 0.9637462235649547
Plot saved to D:\DoAn\Video_Anomaly_Detection\plot\fig4\fig2_anomaly_arson_UCF_000052.png


In [9]:
# Add this cell after the fig2 cell

# Create lists to store videos with IoU = 0 and IoU >= 0.8
iou_zero_videos = []
iou_high_videos = []

# Process each video and check IoU values
for i, video_idx in enumerate(video_indices):
    pred = video_scores[i]
    video_anomaly_ranges = labels[i]
    video_name = video_names.get(video_idx, f"Video_{video_idx}")
    
    # Find anomaly regions
    detected_regions, processed_scores, peaks = find_anomaly_regions(pred)
    
    # Calculate IoU
    iou = calculate_iou(video_anomaly_ranges, detected_regions)
    
    # Save videos with IoU = 0 or IoU >= 0.8
    if iou == 0:
        iou_zero_videos.append(f"{video_name} (index: {video_idx})")
    elif 0.8 <= iou <1:
        iou_high_videos.append(f"{video_name} (index: {video_idx}, IoU: {iou:.4f})")

# Write results to file
output_file = os.path.join(output_dir_fig2, "iou_analysis.txt")
with open(output_file, 'w') as f:
    f.write("=== VIDEOS WITH IoU = 0 ===\n")
    if iou_zero_videos:
        for video in iou_zero_videos:
            f.write(f"{video}\n")
    else:
        f.write("No videos with IoU = 0\n")
    
    f.write("\n\n=== VIDEOS WITH IoU >= 0.8 ===\n")
    if iou_high_videos:
        for video in iou_high_videos:
            f.write(f"{video}\n")
    else:
        f.write("No videos with IoU >= 0.8\n")

print(f"Analysis saved to {output_file}")
print(f"Number of videos with IoU = 0: {len(iou_zero_videos)}")
print(f"Number of videos with IoU >= 0.8: {len(iou_high_videos)}")

PeakDetector(height=0.9, threshold=None, distance=None, prominence=1e-05, width=3, wlen=None, rel_height=0.8, plateau_size=None)
PeakDetector(height=0.9, threshold=None, distance=None, prominence=1e-05, width=3, wlen=None, rel_height=0.8, plateau_size=None)
PeakDetector(height=0.9, threshold=None, distance=None, prominence=1e-05, width=3, wlen=None, rel_height=0.8, plateau_size=None)
PeakDetector(height=0.9, threshold=None, distance=None, prominence=1e-05, width=3, wlen=None, rel_height=0.8, plateau_size=None)
PeakDetector(height=0.9, threshold=None, distance=None, prominence=1e-05, width=3, wlen=None, rel_height=0.8, plateau_size=None)
PeakDetector(height=0.9, threshold=None, distance=None, prominence=1e-05, width=3, wlen=None, rel_height=0.8, plateau_size=None)
PeakDetector(height=0.9, threshold=None, distance=None, prominence=1e-05, width=3, wlen=None, rel_height=0.8, plateau_size=None)
PeakDetector(height=0.9, threshold=None, distance=None, prominence=1e-05, width=3, wlen=None, rel